In [0]:
dbutils.widgets.text("source_system", "", "Source System")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("source_file_format", "", "Source File Format")

source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:
# Path existence check now handled in the next cell (fails gracefully if path is missing)
# landing_df=dbutils.fs.ls(f"/Volumes/caresync_databricks/landing/landing-volume/{source_system}/{table_name}/{landing_timestamp}/")

In [0]:
from pyspark.sql.functions import lit, current_timestamp

landing_path = f"/Volumes/caresync_databricks/landing/landing-volume/{source_system}/{table_name}/{landing_timestamp}/"

# Check whether the landing path (or any parent directory) exists; exit gracefully if not
try:
    landing_files = dbutils.fs.ls(landing_path)
except Exception as e:
    if "FileNotFoundException" in str(e) or "does not exist" in str(e).lower() or "not found" in str(e).lower():
        landing_files = None
    else:
        raise

if not landing_files:
    msg = f"No new files have arrived at {landing_path}. Exiting notebook."
    print(msg)
    dbutils.notebook.exit(msg)

if source_file_format == "csv":
    landing_df = spark.read.option("header", True).option("inferSchema", True).csv(landing_path)
else:
    landing_df = spark.read.parquet(landing_path)

updated_landing_df = landing_df.withColumn("landing_timestamp", lit(landing_timestamp)) \
      .withColumn("insert_timestamp", current_timestamp())

display(updated_landing_df)

In [0]:
updated_landing_df.write.mode("append").saveAsTable(f"caresync_databricks.bronze.{table_name}")